# Autoscaling configurations for publication

# Setup, function definitions and trace generation

In [ ]:
# Basic imports
# %matplotlib widget
from numpy import mean  
from ascal import AscalConfig, Ascal
from examples import aws_eu_west_1_c5m5r5
import csv
import random

In [ ]:
from math import sqrt

def sqrt_noise(x):
    delta = int(round(0.2*sqrt(x)))
    noise = random.randint(-delta, delta)
    return noise

def linear_noise(x):
    delta = int(round(0.1*x))
    noise = random.randint(-delta, delta)
    return noise

def generate_trace_file(file="trace.txt", duration=24*60, offset=0, ramp_up=0.25, plateau=0.5, 
                        ramp_down=0.25, start_value=1, end_value=10, noise=lambda x: 0):

    # Segment durations
    ramp_up = int(round(duration * ramp_up))
    plateau = int(round(duration * plateau))
    ramp_down = int(round(duration * ramp_down))
    valley = duration - (ramp_up + plateau + ramp_down)
    assert valley >= 0, "Duration too short for the given ramp/plateau proportions"

    def add_noise(value):
        return value + noise(value)

    values = []

    # Initial valley
    initial_valley = valley // 2
    for _ in range(initial_valley):
        noisy = add_noise(start_value)
        values.append(noisy)

    # Ramp up
    for i in range(ramp_up):
        base = start_value + (end_value - start_value) * i / (ramp_up - 1)
        noisy = add_noise(round(base))
        values.append(noisy)

    # Plateau
    for _ in range(plateau):
        noisy = add_noise(end_value)
        values.append(noisy)

    # Ramp down
    for i in range(ramp_down):
        base = end_value - (end_value - start_value) * i / (ramp_down - 1)
        noisy = add_noise(round(base))
        values.append(noisy)

    # Final valley
    for _ in range(valley - initial_valley):
        noisy = add_noise(start_value)
        values.append(noisy)

    # Save CSV
    values = values[-offset:] + values[:-offset]  # Apply offset
    with open(file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["value"])
        for v in values:
            writer.writerow([v])


In [ ]:
# Configuration files defining ASCAL problems
config_files = [
    "config_hpa_ca.yaml", 
    "config_fcma_reactive.yaml", 
    "config_fcma_predictive.yaml",
    "config_hpa_ca_fcma_reactive.yaml",
    "config_hpa_ca_fcma_predictive.yaml"
    ]
config_files_1h = [
    "config_hpa_ca_1h.yaml", 
    "config_fcma_reactive_1h.yaml", 
    "config_fcma_predictive_1h.yaml",
    "config_hpa_ca_fcma_reactive_1h.yaml",
    "config_hpa_ca_fcma_predictive_1h.yaml"
]
log_file = None

In [ ]:
# Generate the trace files
random.seed(0)
generate_trace_file(file="trapezoid1.csv", duration=24*60, offset=0, ramp_up=0.25, plateau=0.3, 
                    ramp_down=0.25, start_value=1, end_value=10, noise=sqrt_noise)
generate_trace_file(file="trapezoid2.csv", duration=24*60, offset=100, ramp_up=0.4, plateau=0.4, 
                    ramp_down=0.2, start_value=2, end_value=20, noise=sqrt_noise)
generate_trace_file(file="triangle1.csv", duration=1*60, offset=0, ramp_up=0.5, plateau=0, 
                    ramp_down=0.5, start_value=1, end_value=10, noise=sqrt_noise)
generate_trace_file(file="triangle2.csv", duration=1*60, offset=100, ramp_up=0.4, plateau=0.4, 
                    ramp_down=0.2, start_value=2, end_value=20, noise=sqrt_noise)

# Run experiments and generate data file for plotting (run only once)

Once this cell has been executed, the data file `pub1_results.parquet` will be created in the current directory. This file contains all the necessary data to generate the plots included in the publication, and there is no need to run this cell again.

In [ ]:
import pandas as pd
import re

all_data = []

# Simulate each configuration file
for config_file in config_files + config_files_1h:
    # Extract experiment name and duration from config file name
    match = re.match(r"config_([a-zA-Z0-9_]+?)(_1h)?\.yaml", config_file)
    if not match:
        print(f"Could not parse filename: {config_file}")
        continue
    
    experiment_name = match.group(1)
    duration = "1h" if match.group(2) else "1day"

    # Read the problem configuration file and validate it
    ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)

    # Create the autoscaling problem
    ascal_problem = Ascal(ascal_config, log=log_file)

    # Run the autoscaling problem until the end
    ascal_problem.run()

    # --- Performances ---
    performances = ascal_problem.get_performances()
    for app_name, values in performances.items():
        for time, value in enumerate(values):
            all_data.append({
                "experiment": experiment_name,
                "duration": duration,
                "metric": "performance",
                "name": app_name,
                "time": time,
                "value": value
            })

    # --- Costs ---
    costs = ascal_problem.get_cluster_cost()
    for time, value in enumerate(costs):
        all_data.append({
            "experiment": experiment_name,
            "duration": duration,
            "metric": "cost",
            "name": "Total Cost",
            "time": time,
            "value": value
        })

    # --- Queue Waiting Times ---
    queue_waiting_times = ascal_problem.get_queue_waiting_times()
    for app_name, values in queue_waiting_times.items():
        for time, value in enumerate(values):
            all_data.append({
                "experiment": experiment_name,
                "duration": duration,
                "metric": "qwt",
                "name": app_name,
                "time": time,
                "value": value
            })

# Create the DataFrame
df = pd.DataFrame(all_data)
df.to_parquet("experiments.parquet")

# Plotting results from data file

In [1]:
import pandas as pd
df = pd.read_parquet('experiments.parquet')
df['name'] = df['name'].replace({'Total Cost': 'Cost'})

In [4]:
# Dict of acronyms for experiments
experiment_acronyms = {
    "hpa_ca": "H",
    "fcma_reactive": "R-hu",
    "fcma_predictive": "P-hu",
    "hpa_ca_fcma_reactive": "HR-hu",
    "hpa_ca_fcma_predictive": "HP-hu"
}

In [3]:
# Definition of the plotting function
import matplotlib.pyplot as plt

def plot_experiment(df, experiment, duration, metric, app_label="Application", infobox_text=None, ax=None):
    # Filtrar el DataFrame
    df_filtrado = df[
        (df['experiment'] == experiment) &
        (df['duration'] == duration) &
        (df['metric'] == metric)
    ]

    # Crear la figura para el plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 7))

    # Dibujar una línea para cada aplicación
    for app_name in df_filtrado['name'].unique():
        df_app = df_filtrado[df_filtrado['name'] == app_name]
        ax.plot(df_app['time'], df_app['value'], label=app_name)

    display_metric = metric.upper() if metric == "qwt" else metric.capitalize()
    display_duration = "HRV" if duration == "1h" else "LRW"
    # Añadir títulos y etiquetas
    ax.set_title(f'{app_label} {display_metric} for experiment {experiment_acronyms[experiment]} ({display_duration})')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'{display_metric}')
    if metric != "cost":
        ax.legend()
    ax.grid(True)
    # Añadir infobox si se proporciona texto
    if infobox_text:
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.98, 0.1, infobox_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='bottom', horizontalalignment='right', bbox=props)

    return ax


In [5]:
# Generate all plots

def get_avg_qwt_per_app(df, experiment, duration):
    df_qwt = df[
        (df['experiment'] == experiment) &
        (df['duration'] == duration) &
        (df['metric'] == "qwt")
    ]
    avg_qwt_per_app = {}
    for app_name in df_qwt['name'].unique():
        avg_qwt = df_qwt[df_qwt['name'] == app_name]['value'].mean()
        avg_qwt_per_app[app_name] = avg_qwt
    return avg_qwt_per_app

for experiment in df['experiment'].unique():
    for duration in ["1h", "1day"]:
        height_ratios = [2, 2, 1.2]
        fig, axs = plt.subplots(3, 1, figsize=(7, 8), gridspec_kw={'height_ratios': height_ratios})
        plot_experiment(df, experiment, duration, "performance", app_label="Application", ax=axs[0], infobox_text=None)
        plot_experiment(df, experiment, duration, "cost", app_label="Cluster", ax=axs[1], infobox_text="Total cost = ${:.3f}".format(df[(df['experiment'] == experiment) & (df['duration'] == duration) & (df['metric'] == "cost")]['value'].sum()/3600))
        avg_qwt_per_app = get_avg_qwt_per_app(df, experiment, duration)
        infobox_text = "\n".join([f"{app} Avg. QWT: {avg_qwt_per_app[app]:.3f}" for app in avg_qwt_per_app])
        plot_experiment(df, experiment, duration, "qwt", app_label="Application", ax=axs[2], infobox_text=infobox_text)
        plt.tight_layout()
        plt.savefig(f'plot_{experiment}_{duration}.png')
# Evitar que se muestren en el notebook
plt.close('all')

Horizontal Autoscaling

![](plot_hpa_ca_1day.png) ![](plot_hpa_ca_1h.png)

Reactive history-unaware autoscaling

![](plot_fcma_reactive_1day.png) ![](plot_fcma_reactive_1h.png)

Predictive history-unaware autoscaling

![](plot_fcma_predictive_1day.png) ![](plot_fcma_predictive_1h.png)

Mixed horizontal and reactive history-unaware autoscaling

![](plot_hpa_ca_fcma_reactive_1day.png) ![](plot_hpa_ca_fcma_reactive_1h.png)

Mixed horizontal and predictive history-unaware autoscaling

![](plot_hpa_ca_fcma_predictive_1day.png) ![](plot_hpa_ca_fcma_predictive_1h.png)